# Final Project: Predicting Stock Returns

> **This problem is close to impossible. You are graded on how you think, not on whether you win.**

## The task
- **The required task:** predict the **next-day return** of **AAPL**, using prices up to and including the day before.
  - This is a regression: predict the return itself.
  - Everyone does it, and it is what the required 200 points grade.
- **Put it first.** The first part of your notebook is this regression, done with the full Recipe.
  - Everything else (reframing, strategies, RL) comes after it, under its own heading.
- **Above and beyond (up to 100 more points):** devise and test a trading strategy with whatever choices you like.
  - You choose what to predict or decide.
    - Examples:
      - up or down;
      - five return classes;
      - which days are worth taking risk on;
      - an agent that picks positions.
  - It may trigger only under narrow conditions, if you say which and count the events.
  - Your regression results may be what motivates it, but the regression must still be delivered.

## What this project tests
- Whether you apply the **Recipe for Machine Learning** systematically on a hard problem.
- Your **neural-network** skills: time is handled by your architecture (RNN, LSTM, GRU, causal convolution).
- Optionally, your **reinforcement-learning** skills.
- Your hypotheses, experiments, error analysis, and how failure changes your next step.

## Our philosophy
> **You may outsource the work to an agent. You cannot outsource the responsibility or the learning.**

- You answer for every line, claim, and number in your notebook.
- Agents, libraries, and public data are allowed.
- **No magic:** if you use a model, you must understand it well enough to explain why it should work here.
- A component you cannot explain earns no credit.

## Expect to fail, and be suspicious of success
- Beating the market one day ahead is very unlikely.
- **If your result is so good that you would quit school to trade, you almost certainly cheated by accident.**
  - Common causes: using future data, a leaky split, a model that can look things up online.
- Before you believe a good result, test it with an **honest backtest** of a trading strategy built on it.
- A clear, honest "I could not beat the baseline" can earn a high grade.

## The notebook is a movie, not a snapshot
- Keep your failed experiments in the notebook.
- For each experiment, tell us:
  - why you thought it would help;
  - what happened;
  - what you learned;
  - what you tried next because of it.
- We can only judge your thinking from what you write.
- Good experiments leave evidence that you did not work backwards from the data.
  - Write what you expect, and what would change your mind, before you run it.
  - Change one thing at a time, so you can tell what helped.
  - Compare against the same baseline and the same budget each time.

## The data
- `Data/train/`: one CSV per ticker, daily, 2005 to 2023.
- `Data/sample/`: 2024. This is your **test year**: report your final test result on it, once. **Never train or tune on it.**
  - "Once" means one final model, chosen without looking at 2024.
  - The given summary also scores 2024. That is allowed.
  - Do not iterate on what it prints.
  - The hidden period comes after 2024.
  - Do not add files to `Data/`: the summary checks that it is unchanged.
- Columns, in this order in the files: `Dt, Adj Close, Close, Div, Factor, High, Low, Open, Volume`.
- **`Close` is the raw traded price. It is not adjusted.** Decide how to handle that, and justify your choice.
- `Adj Close` is adjusted after the fact (for splits and dividends). You cannot trade at it.
- `Factor` is the split ratio on a split day (a few days carry other distribution events), otherwise 1.0.
  - The summary counts every day with `Factor` above 1.0001 as an event.
  - The key is called `splits_detected` for historical reasons. Report that count, not your own opinion of which events are "real" splits.
- `Div` is the cash dividend per share on its ex-date, in the same raw units as `Close`.
- `Volume` is raw: it is not adjusted for splits.
- Tickers include AAPL, other stocks, SPY, and sector ETFs. Some start trading late.
  - Their early rows are present but empty (missing values). A **late listing** is a ticker with such rows.
- Real data is messy. Look at it before you trust it.
- You may add **public external data** (for example earnings dates). Cite the source. Respect the rule below on the future.

In [ ]:
# Setup (given). Loads every ticker's daily CSV exactly as provided: nothing is adjusted or filled.
import pathlib
import numpy as np
import pandas as pd
from evaluate_strategy import evaluate_strategy  # given; see "If you submit a strategy"

DATA = pathlib.Path("Data")
TICKERS = sorted(p.stem for p in (DATA / "train").glob("*.csv"))
train = {t: pd.read_csv(DATA / "train" / f"{t}.csv", parse_dates=["Dt"]) for t in TICKERS}
sample = {t: pd.read_csv(DATA / "sample" / f"{t}.csv", parse_dates=["Dt"]) for t in TICKERS}  # mock holdout: never train on it
print(len(TICKERS), "tickers:", TICKERS)
print("train rows per ticker:", len(train["AAPL"]), " sample rows:", len(sample["AAPL"]))


## The hidden test
- We will run your final model on **every day of a later period you have never seen**, one day at a time.
- That data will contain kinds of events that also appear in your training data (for example, splits).
- Your pipeline must handle them **automatically, with no manual fixes**.
- Defending against events your training data never showed is optional (see "above and beyond").
- The same transformations must apply identically to training, validation, test, and hidden data.
- This test checks that your reported results are honest. It does not rank you by profit.

## The rule on the future
- A prediction for day $t$ may use only data up to and including day $t-1$.
- Leaks are often invisible. For example:
  - standardizing with the mean and standard deviation of the **whole** training set gives early dates knowledge of later dates.
  - the same holds for imputation, PCA, feature selection, and a scaler fit on all the data.
  - rolling features can straddle your train/validation boundary: leave a gap at least as long as your longest lookback window, counted in trading days (rows).
  - Windows that reach back into the gap or train rows are fine for prediction. Say what you did.
  - At test time `predict_next` sees all earlier history, so test windows may reach into 2023.
- Use statistics computed from the past only (expanding or rolling).
- A pretrained model may have seen the dates you predict. If you use one, state its training cutoff and test only after it.
- Your final model must run **offline**. No web search, hosted model, or other tools at prediction time.
  - A small model run locally has no tools. The risk is code that calls a web service or a hosted model.
  - We will run your model with the network switched off. A model that needs the network gets no credit.

## Required: describe the shape of your examples
- One section, easy to find, with a clear heading.
- State the shape of **one example** and name every axis.
  - Example: `(time, feature)`, with `time` = the last 60 days and `feature` = open, high, low, close, volume.
- Name and explain every feature, especially ones you created.
- State the target.
- Every model in the notebook must say which format it uses.
- We use this to understand your thinking.
  - A sequence of feature vectors fits a sequence model (RNN, LSTM, GRU, causal convolution).
  - Using the row index of a table as a stand-in for time is a problem.

## What your notebook must contain
1. **Problem and metric:** your task, your metric, and why. Name the baseline you must beat.
2. **Data checks:** what you found (splits, missing values, late-listed tickers), and what you did.
3. **EDA:** each plot should lead to a decision. Tell us the decision.
4. **Split design:** chronological train, gap, validation, gap, test. Assert that the dates do not overlap.
   - The test period is the 2024 sample year.
   - You may refit on train plus validation before the test, if you say so.
5. **Example format:** the section above.
6. **Baselines first:** predict zero, and one simple model (linear or moving average).
7. **At least three neural-network models.** Say your hypothesis before each one.
8. **Error analysis:** it must lead to your next experiment. Show the link.
9. **Out-of-sample honesty:** your test result, the baseline, and a confidence interval. Is the difference real?
10. **Final model:** see below.
11. **Submission Summary:** the last two cells (your `SUMMARY`, then the given print cell). See below.

## Your final model
- Put it in its own file, `final_model.py`, plus a weights file.
- **Packages:** we run it in one fixed environment with numpy, pandas, scipy, scikit-learn and PyTorch.
  - Anything else may not load, and then you lose the "cannot load" points.
  - Colab lets you install more. We cannot.
  - Your notebook must run on a free Colab CPU.
  - Only `final_model.py` is limited to the packages above.
  - Save weights in a portable format (`state_dict`, `.npz`), not a pickled model object.
- It defines `predict_next(history)`.
  - `history`: a dict, ticker -> pandas DataFrame, with the same columns as the CSV files.
    - All 14 tickers, sorted by date, with rows up to and including day $t-1$ only.
    - The history includes 2024. Late-listed tickers keep their empty rows.
  - It returns your prediction for day $t$.
- It must load its own weights and run without your notebook.
- We will import it and run it automatically. If it cannot load, you lose points and we will not debug it.
- **Limits:** each call must return within 30 seconds, and the whole hidden run is limited to 60 minutes.
  - The hidden period is a few hundred trading days (about two years), one call per day.
  - Use plain, long-standing pandas, numpy and PyTorch features: the grader's package versions can differ slightly from yours.
  - It returns your prediction of AAPL's next-day **simple return**, $(\text{Close}\cdot\text{Factor}+\text{Div})/\text{previous Close}-1$, as a number.
  - If you model another quantity (log return, scaled return), convert it before you return it.
- `final_model.py` runs offline with the packages above, so it cannot download a pretrained model.
  - It contains your own small network, and its weights are at most 25 MB (see "What you hand in").
  - A pretrained model may be used in notebook experiments. Name it with a pinned revision and state its training cutoff.

## If you add your own objective (above and beyond)
- Devising and testing a strategy of your own is allowed, and often wise. It is **in addition to** the required regression.
- Example: a reinforcement-learning agent that chooses positions has moved from "predict" to "devise a strategy".
- Set `kind` to `"extra_objective"` in `SUMMARY["objective"]`. The regression summary lines still print for your required part.
- **You must say what you are doing clearly, at the start of the part that follows your regression:**
  - the old objective and the new one;
  - why you changed, citing the experiment that made you change;
  - the metric for the new objective, and the baseline it must beat.
- **You must provide an automatic test:** a function `metric_test`, which you put in `SUMMARY["objective"]`.
  - It computes the new metric for your model and for the baseline on the 2024 sample year.
  - It returns a dict with `model`, `baseline`, `n` and `higher_is_better`.
  - The dict also holds `ci_low` and `ci_high`: the interval of model minus baseline.
  - The summary runs it and prints the result, so we can grade the new objective from the printed lines.
  - The summary calls `metric_test()` with **no arguments**, from the folder that holds `Data/`. It must load what it needs from `Data/` itself.
  - Graders may call it again without your notebook.
  - Define it in `final_model.py` or in a helper module you hand in.
  - The helper need not be imported by `final_model.py`.
  - Put your strategy rule in that module too, so the test is reproducible from the hand-in files alone.
- Without a clear statement and a test, we cannot grade the new objective, and it earns no extra credit.
- Your required regression is graded the same way whether or not you add an objective.
- If you try several strategies, name **one as your primary** before you look at the test period, and count them all.
  - Every extra try is another chance of a lucky result.
  - More good work earns more, if you count it: extra models, error analyses and strategies of high quality earn depth credit.
  - Weak extras earn nothing.

## If you submit a strategy
- Your strategy is a table of positions: one row per day, one column per ticker.
  - A value is a fraction of your equity: -1 fully short, 0 flat, +1 fully long. No leverage:
    - larger values are clipped;
    - if the sum of the absolute positions exceeds 1, it is scaled down to 1.
  - Row $d$ is your decision at the close of day $d$, using data up to day $d$. It earns day $d+1$'s return.
  - The first row earns nothing for itself: to earn the first 2024 day, start your table on the last day before 2024.
- Pass those positions to the **given** function `evaluate_strategy` in `evaluate_strategy.py` (we supply it, so do not rewrite it).
  - It applies that one-day lag and a fixed trading cost (5 basis points per unit traded) for you.
  - It prints the same statistics for everyone, so graders can compare.
- It prints:
  - cumulative return, and the same for buy-and-hold;
  - buy-and-hold is an equal-weight basket of the tickers in your positions table;
  - maximum drawdown;
  - lowest equity: the smallest value your equity reaches, with a start of 1.0;
  - number of trades, time in the market, and turnover.
- **Trading stops if your capital reaches zero.** The function prints the date, and the results after it do not count.
- **How many events did your strategy see?**
  - The function prints the number of completed trades in each year.
  - You need enough events to at least hint that the result is not gross luck. You cannot prove how likely luck is.
  - How many is enough depends on the length of the period. A strategy that trades rarely may have too few to judge.
  - If so, say so; that is an honest finding.
  - More tickers and walk-forward folds give more events. Note that tickers move together, so they are not fully independent.
  - The function also prints a randomized-timing comparison: the same number of trades at random times.

## Going above and beyond
- Grading: **200 required points, plus up to 100 for going above and beyond.**
- Ideas (the 100 rewards depth and sound reasoning, not luck):
  - the main route: **devise and test a strategy** of your own design.
    - The cells below say how to declare it and test it honestly.
  - a well-justified reframing of the problem, as part of that;
  - hypotheses that hold only under conditions you identify;
  - using the **other tickers** with a purpose: the file holds more than AAPL for a reason.
    - to stimulate your thinking: can they help predict AAPL, or show where and why prediction is easier or harder?
  - a pipeline that also survives events your training data never showed, with a test that proves it
  - an **honest backtest** of a strategy: state the signal lag, include costs, fix your rule before the test period, compare with buy-and-hold
  - a **reinforcement-learning** formulation: define state, action, reward and costs, and avoid leakage;
    - to stimulate your thinking: does your agent's choice of position change its next state? Why does the cost term matter?
    - how much does the result vary across random seeds?
    - could your agent exploit the simulator, for example by acting on the same-day close? Show how you blocked it.
  - a **small pretrained language model** (SmolLM2-135M or SmolLM2-360M only), with a clear purpose;
    - it is a notebook experiment: `final_model.py` cannot include it (offline, 25 MB, fixed packages);
    - compare it with a non-LLM network of equal budget;
    - to stimulate your thinking: turning numbers into text is a questionable way to feed a language model.
    - what if you replace the input layer with a learned projection of your feature vectors and put a small head on top?
    - for example, a head that chooses an action: hold, increase (even from zero), or decrease (even go short).
    - does that beat a plain GRU of equal size?
    - is it much better before its training cutoff than after? That suggests it memorized prices.
    - does public text (cited, timestamped before the prediction) add anything beyond the numbers?
  - other model types, if you can explain why they fit.
- Everything must run on a free Colab CPU. A model large enough to need a GPU would overfit this noisy problem.

## What you hand in
- **One archive** (`.zip`), submitted on the Course Management System. The deadline is in the announcement.
- Inside it:
  - your notebook `Stock_Prediction_Assignment.ipynb`, **with outputs saved**;
    - we read it; we do not re-run it;
  - `final_model.py`;
  - its weights, **at most 25 MB in total**;
    - file names start with `final_model` (for example `final_model.npz`);
  - any small helper module that `final_model.py` imports;
  - your `metric_test` module, if you did the extra part.
    - Only weights files must start with `final_model`.
- Follow the **Course Assignment Guidelines** (naming of the archive, what to include).
  - Leave out the given files (`evaluate_strategy.py`, `submission_summary.py`): we have them.
  - Keep the archive small: remove backups, checkpoints (`.ipynb_checkpoints`), `__pycache__`, and anything we do not need.
  - Do not include the `Data/` folder: we already have it.

## Why the Submission Summary exists
- The last cell prints a fixed-format summary. It saves us from hunting through your notebook.
- Run it last, after everything else. A missing or unrun summary costs points.
- It prints, among other things:
  - your example format next to the real shape of an example;
  - your split dates and your baseline and model scores;
  - whether your model works from returns, not memorized price levels or dates;
    - we multiply prices and dividends by 3 and shift dates by ten years.
    - A sound model barely changes.
  - how long a prediction call takes;
  - the source of your functions, and your short written report.
- You fill in the written report. Everything else prints automatically.
- Reading the printed lines:
  - `distinguishable: yes` only means the interval excludes 0. Check `model better:` to see which way.
  - The lag test prints your return next to the return with the signal lagged one more day.
    - A big gain from lagging is a warning.
    - A small difference either way is noise.
  - The randomized-timing comparison shifts your held positions in time and reports the share of shifts you beat.

## Submission Summary (given; run last)
- **Step 1.** Fill in the `SUMMARY` dict in the next cell. Do not rename its keys.
  - `example` is ONE real example, exactly as it is fed to your network (the array or tensor, not the raw history).
  - `test` holds one loss value per test example, for your model and for both baselines, in the same order.
    - Use the squared error of the return (in return units, not percent), unless you chose another metric.
    - `loss_simple` is the one simple baseline you named in section 6.
  - `strategy` is `None`, or `dict(positions=<your positions table>, prices=sample)`.
  - `functions` lists your own functions or classes.
    - We print their source, at most 150 lines each.
  - `data_events`: count over ALL tickers in `Data/train/`, not only AAPL.
  - `report`: `key_line` is one sentence (at most 300 characters) with your main finding. The other report fields are cut at 1500 characters.
- **Step 2.** Run the given cell after it, last. It prints lines that start with `SUMMARY|`.
  - Keep the given cells. Put your own work **before** this section, and you may delete instruction cells.
  - Run the notebook from the folder that holds `Data/`, `evaluate_strategy.py`, `submission_summary.py` and `final_model.py`.
  - On Colab, upload those files first.
- A missing or unrun summary costs points.

In [ ]:
# Fill in (yours). Keep every key; put None where a part does not apply.
SUMMARY = dict(
    seeds=None,  # e.g. {"numpy": 0, "torch": 0}
    objective=dict(kind="predict_next_return",  # or "extra_objective": fill every field; metric_test is a function returning dict(model, baseline, n, higher_is_better, ci_low, ci_high)
                   old=None, new=None, why=None, metric=None, baseline=None, metric_test=None),
    example_format=dict(axes=(), shape=(), features=[], target=""),  # e.g. axes=("time", "feature"), shape=(60, 5)
    example=None,  # ONE real example, exactly as fed to your final model
    splits=dict(train=("YYYY-MM-DD", "YYYY-MM-DD"), val=("YYYY-MM-DD", "YYYY-MM-DD"), test=("YYYY-MM-DD", "YYYY-MM-DD")),
    data_events=dict(splits_detected=None, late_listings_detected=None, treatment=""),
    test=dict(metric_name="", loss_model=None, loss_zero=None, loss_simple=None),  # one loss per test example, same order
    strategy=None,  # or dict(positions=<DataFrame>, prices=sample)
    functions=[],  # your own functions, e.g. [make_features, my_model_class]
    pretrained=None,  # or dict(name=..., revision=..., n_params=..., cutoff=...)
    report=dict(key_line="", what_changed="", what_tried="", what_failed=""),
)


In [ ]:
GIVEN = {'evaluate_strategy.py': '0f02fe6fd4bf97f8', 'submission_summary.py': '0f6e429fc55eb2a1', 'Data': '0a6e79446d953936'}  # fingerprints of the given files
# Given: run this LAST. Do not edit.
from submission_summary import print_summary
print_summary(SUMMARY, train, sample, given=GIVEN)
